In [3]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/train_reviews_clean.csv"
)

print(df.shape)
df.head()

(25000, 6)


,review,label,review_length,sentiment,clean_review,clean_length
0,Bromwell High is a cartoon comedy. It ran at t...,1,806,Positive,Bromwell High is a cartoon comedy. It ran at t...,806
1,Homelessness (or Houselessness as George Carli...,1,2366,Positive,Homelessness (or Houselessness as George Carli...,2322
2,Brilliant over-acting by Lesley Ann Warren. Be...,1,841,Positive,Brilliant over-acting by Lesley Ann Warren. Be...,841
3,This is easily the most underrated film inn th...,1,663,Positive,This is easily the most underrated film inn th...,663
4,This is not the typical Mel Brooks film. It wa...,1,647,Positive,This is not the typical Mel Brooks film. It wa...,647


In [4]:
df_small = df.sample(
    n=2000,
    random_state=42
)

print(df_small.shape)

(2000, 6)


In [5]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_small["clean_review"],
    df_small["label"],
    test_size=0.2,
    random_state=42,
    stratify=df_small["label"]
)

print(len(train_texts))
print(len(test_texts))

1600
400


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

print("Tokenizer Loaded")

Tokenizer Loaded


In [7]:
sample_text = train_texts.iloc[0]

tokens = tokenizer(sample_text)

print(tokens.keys())

KeysView({'input_ids': [101, 1045, 2572, 1037, 2502, 5470, 1037, 27498, 6925, 3004, 1998, 1045, 1005, 2310, 2464, 2068, 2035, 1998, 2023, 2003, 2028, 1997, 1996, 2190, 999, 2009, 1005, 1055, 6057, 1010, 6298, 1010, 1998, 1037, 4438, 1012, 1045, 16755, 2023, 2005, 2035, 5535, 1012, 2009, 1005, 1055, 2307, 2005, 2210, 4268, 2138, 2009, 1005, 1055, 2092, 1010, 21686, 1998, 2307, 2005, 6001, 1998, 9458, 2138, 2009, 1005, 1055, 6057, 1998, 2025, 2058, 1996, 2327, 1012, 1045, 3427, 2009, 2043, 1045, 2001, 2210, 1998, 1045, 2145, 3422, 2009, 2085, 1012, 2009, 2038, 2307, 3210, 2008, 2026, 2155, 1998, 1045, 14686, 2035, 1996, 2051, 1012, 1996, 3772, 2003, 2307, 1998, 2009, 2196, 4152, 2214, 1012, 2065, 2017, 2066, 8867, 7122, 1998, 7472, 2015, 2017, 2097, 2293, 2023, 1012, 1045, 1005, 2310, 3427, 2116, 1037, 21686, 3185, 1999, 2026, 2051, 1998, 2023, 2003, 1996, 2190, 1997, 2068, 2035, 1012, 1006, 3374, 6373, 1007, 1045, 3811, 16755, 2023, 3185, 1998, 2035, 1996, 27498, 6925, 3004, 3065, 1012,

In [8]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    test_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

print("Tokenization Complete")

Tokenization Complete


In [9]:
print(len(train_encodings["input_ids"]))
print(len(test_encodings["input_ids"]))

1600
400


In [10]:
import torch

In [11]:
class IMDbDataset(torch.utils.data.Dataset):
    
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):
        return len(self.labels)

In [12]:
train_dataset = IMDbDataset(
    train_encodings,
    train_labels
)

test_dataset = IMDbDataset(
    test_encodings,
    test_labels
)

print(len(train_dataset))
print(len(test_dataset))

1600
400


In [13]:
from transformers import (
    AutoModelForSequenceClassification
)

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("Model Loaded")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Loaded


In [16]:
from transformers import TrainingArguments

In [17]:
training_args = TrainingArguments(
    output_dir="../models/distilbert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    num_train_epochs=1,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    logging_dir="../models/logs",

    logging_steps=100
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [18]:
from transformers import Trainer

In [19]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)

In [20]:
trainer.train()

c:\Users\dr_ra\anaconda3\envs\imdbsentiment\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.419566,0.488047


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=400, training_loss=0.49703862190246584, metrics={'train_runtime': 3342.3977, 'train_samples_per_second': 0.479, 'train_steps_per_second': 0.12, 'total_flos': 105973918924800.0, 'train_loss': 0.49703862190246584, 'epoch': 1.0})

In [21]:
results = trainer.evaluate()

print(results)

c:\Users\dr_ra\anaconda3\envs\imdbsentiment\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch
0.419566,0.488047,1


{'eval_loss': 0.48804688453674316}


In [22]:
predictions = trainer.predict(test_dataset)

In [23]:
import numpy as np

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = test_labels.values

In [24]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_true,
    y_pred
)

print("DistilBERT Accuracy:", accuracy)

DistilBERT Accuracy: 0.855


In [25]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.87      0.82      0.84       192
           1       0.84      0.89      0.86       208

    accuracy                           0.85       400
   macro avg       0.86      0.85      0.85       400
weighted avg       0.86      0.85      0.85       400



In [26]:
data/processed/train_reviews_clean.csv

NameError: name 'data' is not defined